In [2]:
# -*- coding: utf-8 -*-
"""
茨城県市町村シェープファイル + geopandas から
隣接リスト (ibaraki_neighbors.csv) を自動生成するスクリプト。

前提:
  - panel_ibaraki_ssdse.csv : すでに作成済み
  - 茨城県市町村のポリゴン shapefile がある（全国の市区町村shpでもOK）
  - geopandas がインストールされていること

使い方:
  1. SHAPEFILE のパスだけ自分のファイルに書き換える
  2. python build_neighbors_from_shapefile.py を実行
  3. ibaraki_neighbors.csv が出力される
"""

import os
import re
import pandas as pd
import geopandas as gpd


# ======================================================
# ★ここだけ自分の環境に合わせて書き換える★
# ======================================================

# 1) パネルファイル（前に作ったやつ）
PANEL_FILE = "panel_ibaraki_ssdse.csv"

# 2) シェープファイルのパス（.shp）
#    例: "N03-23_220101.shp" みたいな全国市区町村のやつ or 茨城県だけのやつ
SHAPEFILE = "N03-20240101_08.shp"   # ← ここを自分の .shp に変更


# 3) 茨城県を表す値（県名）
PREF_NAME_IBARAKI = "茨城県"


# 出力する隣接CSV
NEIGH_OUT = "ibaraki_neighbors.csv"


# ======================================================
# ユーティリティ
# ======================================================

def clean_city_name(name: str) -> str:
    """
    市町村名の表記ゆれを軽く吸収するためのクリーニング。
    - 全角/半角カッコ内を削除
    - 前後の空白削除
    """
    s = str(name)
    s = re.sub(r"（.*?）", "", s)
    s = re.sub(r"\(.*?\)", "", s)
    return s.strip()


def auto_detect_pref_city_cols(gdf: gpd.GeoDataFrame):
    """
    shapefile の列から
      - 都道府県名の列
      - 市町村名の列
    をそれっぽい名前で自動推測する。
    """
    cols = list(gdf.columns)
    print("Shapefile columns:", cols)

    pref_col = None
    city_col = None

    # 1) 県名候補: 候補になりそうな名前
    pref_candidates = ["PREF_NAME", "KEN", "PREF", "PREFECTURE", "N03_001", "都道府県名"]
    for c in pref_candidates:
        if c in cols:
            pref_col = c
            break

    # 2) 市町村名候補: "市", "町", "村" を多く含む列
    if pref_col is None:
        print("  -> 都道府県列は自動検出できませんでした。全国shpで N03_001 が県名のことが多いです。")

    # city候補を推測
    text_cols = [c for c in cols if gdf[c].dtype == object]  # 文字列列だけ
    best_col = None
    best_score = -1

    for c in text_cols:
        values = gdf[c].astype(str).head(200)
        # 「市」「町」「村」を含む行の割合を見る
        cnt = values.str.contains("市|町|村", regex=True).sum()
        score = cnt / len(values)
        if score > best_score:
            best_score = score
            best_col = c

    city_col = best_col

    print(f"Auto-detected pref_col = {pref_col}, city_col = {city_col}")
    return pref_col, city_col


# ======================================================
# メイン処理
# ======================================================

def main():
    # 1) パネルの市町村名を読み込んでおく
    if not os.path.exists(PANEL_FILE):
        raise FileNotFoundError(
            f"{PANEL_FILE} が見つかりません。"
            "先に AR+SSDSE のスクリプトを実行して panel_ibaraki_ssdse.csv を作成してください。"
        )

    panel = pd.read_csv(PANEL_FILE)
    panel["city_name"] = panel["city_name"].map(clean_city_name)
    panel_cities = sorted(panel["city_name"].unique())

    print("=== Cities in panel (unique) ===")
    print(panel_cities)

    # 2) shapefile を読み込み
    if not os.path.exists(SHAPEFILE):
        raise FileNotFoundError(
            f"{SHAPEFILE} が見つかりません。\n"
            "SHAPEFILE のパスを正しいものに書き換えてください。"
        )

    print("\n=== Loading shapefile ===")
    gdf = gpd.read_file(SHAPEFILE)

    # 都道府県列 & 市町村列を自動推定
    pref_col, city_col = auto_detect_pref_city_cols(gdf)
    if city_col is None:
        raise ValueError(
            "市町村名の列を自動検出できませんでした。"
            "auto_detect_pref_city_cols の中で列候補を print して確認し、"
            "city_col を手で指定してください。"
        )

    # 3) 茨城県だけに絞る（全国shpの場合）
    if pref_col is not None and PREF_NAME_IBARAKI is not None:
        gdf = gdf[gdf[pref_col] == PREF_NAME_IBARAKI].copy()
        print(f"\nFiltered to {PREF_NAME_IBARAKI}: {len(gdf)} rows")
    else:
        print("\n[Note] 都道府県列が不明なので、全レコードを対象にします。")

    # 4) 市町村名をクリーニング
    gdf["city_name"] = gdf[city_col].map(clean_city_name)

    # 5) パネルに登場する市町村だけ残す
    gdf = gdf[gdf["city_name"].isin(panel_cities)].copy()
    print(f"\nAfter matching with panel cities: {len(gdf)} polygons")
    print(gdf[["city_name"]].head())

    # パネル側にあって shapefile に無かった市町村（表記ズレチェック）
    missing = sorted(set(panel_cities) - set(gdf["city_name"]))
    if missing:
        print("\n[Warning] 以下の市町村は shapefile から見つかりませんでした:")
        for m in missing:
            print("  -", m)
        print("市町村名の表記（旧字/新字、スペース、カッコなど）を確認してください。")

    # 6) 空間インデックスを使って隣接判定
    print("\n=== Building neighbor list via geometry.touches ===")
    gdf = gdf.reset_index(drop=True)
    sindex = gdf.sindex

    neighbors = set()  # (city_i, city_j) のタプル

    for i, geom_i in enumerate(gdf.geometry):
        name_i = gdf.loc[i, "city_name"]

        # バウンディングボックスで候補を絞る
        possible_idx = list(sindex.intersection(geom_i.bounds))
        possible_idx = [j for j in possible_idx if j != i]

        for j in possible_idx:
            geom_j = gdf.geometry.iloc[j]
            name_j = gdf.loc[j, "city_name"]

            # touches: 境界が接していて、内部は重なっていない
            if geom_i.touches(geom_j):
                pair = tuple(sorted([name_i, name_j]))
                neighbors.add(pair)

    # 隣接リスト DataFrame に
    rows = []
    for ci, cj in sorted(neighbors):
        rows.append({"city_name": ci, "neighbor_name": cj})
        rows.append({"city_name": cj, "neighbor_name": ci})  # 両方向

    neigh_df = pd.DataFrame(rows).drop_duplicates().sort_values(
        ["city_name", "neighbor_name"]
    )

    print("\nSample of neighbor pairs:")
    print(neigh_df.head(20))

    neigh_df.to_csv(NEIGH_OUT, index=False, encoding="utf-8-sig")
    print(f"\nSaved neighbor list to {NEIGH_OUT}")


if __name__ == "__main__":
    main()


=== Cities in panel (unique) ===
['かすみがうら市', 'つくばみらい市', 'つくば市', 'ひたちなか市', '下妻市', '五霞町', '八千代町', '利根町', '北茨城市', '取手市', '古河市', '土浦市', '坂東市', '城里町', '境町', '大子町', '大洗町', '守谷市', '小美玉市', '常総市', '常陸大宮市', '常陸太田市', '日立市', '東海村', '桜川市', '水戸市', '河内町', '潮来市', '牛久市', '石岡市', '神栖市', '稲敷市', '笠間市', '筑西市', '結城市', '美浦村', '茨城町', '行方市', '那珂市', '鉾田市', '阿見町', '高萩市', '鹿嶋市', '龍ケ崎市']

=== Loading shapefile ===
Shapefile columns: ['N03_001', 'N03_002', 'N03_003', 'N03_004', 'N03_005', 'N03_007', 'geometry']
Auto-detected pref_col = N03_001, city_col = N03_004

Filtered to 茨城県: 220 rows

After matching with panel cities: 220 polygons
  city_name
0       水戸市
1       日立市
2       日立市
3       日立市
4       日立市

=== Building neighbor list via geometry.touches ===

Sample of neighbor pairs:
   city_name neighbor_name
0    かすみがうら市           土浦市
2    かすみがうら市          小美玉市
4    かすみがうら市           石岡市
6    かすみがうら市           美浦村
8    かすみがうら市           行方市
10   かすみがうら市           阿見町
12   つくばみらい市          つくば市
14   つくばみらい市      